# Lab: Colorectal Cancer Histology with PyTorch
*Version 4.0, Summer Semester 2025*

Based on the learnings of the previous lab, you will now work on a convolutional neural network and apply it to a scenario related to healthcare.

### The Dataset

We'll use the [Colorectal Cancer Histology](https://zenodo.org/record/53169#.XGZemKwzbmG) dataset. It was the basis of an article published to *Nature* in 2016 and is [available for free through Open Access](https://www.nature.com/articles/srep27988). Kather et al. achieved an accuracy of 87.4% for a multiclass scenario. Let's see how far you can get. The dataset was then published under Creative Commons license and added to the [TensorFlow example database](https://www.tensorflow.org/datasets/catalog/colorectal_histology), making it easier for us to load and process the data - even though it involves a little step to convert that to PyTorch; but that's pre-coded for you.

The dataset includes 5,000 RGB histological images, each 150x150px. These have been classified for 8 different targets (labeled as: 'tumor', 'stroma', 'complex', 'lympho', 'debris', 'mucosa', 'adipose', 'empty'). The following image contains representative images of these classes:

![Representative images](https://github.com/andijakl/MachineLearning/raw/main/lab%203%20-%20deep%20learning%20-%20colorectal%20cancer/lab3b-representative-images.jpg) *(a) tumour epithelium, (b) simple stroma, (c) complex stroma (stroma that contains single tumour cells and/or single immune cells), (d) immune cell conglomerates, (e) debris and mucus, (f) mucosal glands, (g) adipose tissue, (h) background.*

The dataset is around 260 MB, which still makes it possible to use a standard laptop without dedicated hardware acceleration for machine learning.

## Your Name

Enter your name in the block below:

In [ ]:
# @title Enter your name here {"run":"auto","vertical-output":true}
student_name = '' # @param {type:"string"}
import uuid
import hashlib
import os
from datetime import datetime

notebook_version = 4.0

def getData():
    mid = hashlib.sha256(str(uuid.getnode()).encode()).hexdigest()[:10]
    execution_time = datetime.now().isoformat()
    return mid, execution_time

mid, execution_time = getData()

# Store metadata in the notebook itself
print(f"Your name: {student_name} ({notebook_version} - {mid} - {execution_time})")


The blocks with assert perform automated tests so that you know if your solution is correct, without giving away how to code it. Simply execute these blocks. If you don't get any output from them, you know that your code is correct.

In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert student_name != '', "Please enter your name in the block above"

## 1: Initializing

In this lab, the imports you will need are pre-defined:

In [ ]:
# Predefined imports - just run this cell
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline

import unittest
test_case = unittest.TestCase()

For the dataset we are going to use, the CPU is still OK, but you will see benefits from using the GPU. Write the code snippet to see if a GPU is available. In Google CoLab, you can switch your runtime type to a GPU for free for a limited time per day.

Use the standard code from `torch.cuda.is_available()` to set a `device` variable accordingly so that we can later move the model and the data to the GPU.

In [ ]:
# Check if GPU is available and store it in a variable called device
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert device is not None, "Please initialize the device variable."

In many real-life cases, you will have a lot of data or an otherwise dynamic process to feed data into your neural network.

Both PyTorch and TensorFlow contain sample datasets that make it easier to work with. Both frameworks are available for Google Colab. In this lab, we'll use a dataset from TensorFlow, as PyTorch doesn't contain medical image classification samples.

TensorFlow contains a module called `tensorflow_datasets` ([tfds](https://www.tensorflow.org/datasets/api_docs/python/tfds)), which can automatically download data and provide it in a suitable format for further processing by TensorFlow. Execute the following code block to import the module:

*Note:* if not running on Google CoLab, you might need to install `tensorflow-datasets` via pip in version 4 or higher.

In [ ]:
# Pre-defined code: simply execute this cell
import tensorflow as tf
import tensorflow_datasets as tfds
print(f"Running tfds version: {tfds.__version__}")

The example we're looking for is called `colorectal_histology`. The fact that it's contained in `tfds` means that we do not need to worry about downloading the data manually. It also takes care of loading all the individual image files into our input pipeline.

*Note:* the `colorectal_histology_large` dataset is different – it contains 10 high-res 5000x5000px images, each containing more than one tissue types. This would then not just require a classification of the whole tissue sample as we're doing, but additionally a localization of where to find the specific classes.

## 2: Load & Analyze the Dataset

Now, it's time to load the dataset. In this dataset, all 5,000 examples come in one piece – there is no pre-defined train / test / validation split. Therefore, we need to define the split manually.

We use `tfds` to `load` the dataset. It has four parameters:

* First, the dataset name as a string – simply provide the name of the dataset we want to load. Copy and paste the name of our histology dataset from the available datasets we printed before.
* Second, the `split` to use. This dataset provides all samples in a `train` variable. Based on this, we want to use create two sub-dataset variants: 80% for training (`ds_train`), 20% for testing (`ds_test`). Specify this split using `['train[:80%]', 'train[80%:]']` that you assign to the `split` parameter.
* Third, set `as_supervised` to `True`. This automatically creates a dataset that contains the input data and the labels as a python tuple. This means that both are contained in a single returned item.
* To additionally take a look at further information about the dataset, also set `with_info` to `True`

The `tfds.load` function returns two data structures. Simply seperate these with a comma:

1. A **tuple** that contains **both datasets**, according to the split. Name them `ds_train` and `ds_test` accordingly. A Python tuple is written like this: `(ds_train, ds_test)`
2. The **info about the dataset** which we requested to load. Store this in a variable `ds_info`.

In total, we therefore have 3 distinct variables that the `load` function returned.

In [ ]:
# Import the colorectal_histology dataset into the variables: (ds_train, ds_test), ds_info
# Pre-defined code: simply execute this cell
(ds_train_tf, ds_test_tf), ds_info = tfds.load('colorectal_histology',
                                           split=['train[:80%]', 'train[80%:]'],
                                           as_supervised=True,
                                           with_info=True)

In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert ds_train_tf.element_spec[0].shape == [150, 150, 3]
assert ds_test_tf.element_spec[0].shape == [150, 150, 3]
assert len(list(ds_train_tf)) == 4000
assert len(list(ds_test_tf)) == 1000
assert isinstance(ds_train_tf, tf.data.Dataset)
assert type(ds_info) == tfds.core.dataset_info.DatasetInfo
assert ds_info.name == 'colorectal_histology'

Next, print the contents of the `ds_info` variable to get some basic info about the dataset.

In [ ]:
# Print the dataset info
raise NotImplementedError


The item `featues["label"]` of `ds_info` contains a bit more information about the target labels we want to classify for. First, assign its `num_classes` property to a variable called `class_count` and also print this new variable you just created:

In [ ]:
# Assign num_classes contained in ds_info to a variable class_count and print it
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert class_count > 0


You can also retrieve the short target names for each class. To get more information about these, refer to the [original paper](https://www.nature.com/articles/srep27988). Assign the property `names` of `features["label"]` from your dataset info (`ds_info`) to a new variable `class_names` and also print it:

In [ ]:
# Assign the class names to a variable class_names and print it
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert class_names[0] == 'tumor'
assert class_names[7] == 'empty'
assert len(class_names) == 8

Now, just execute the following pre-defined code to convert the data from TensorFlow to PyTorch tensors, which we can then access with the `DataLoader` class from PyTorch.

In [ ]:
# Predefined code - just execute
class ColorectalHistologyDataset(Dataset):
    def __init__(self, tf_dataset):
        self.data = []
        for image, label in tf_dataset:
            # Convert image and label to PyTorch tensors
            image = torch.from_numpy(image.numpy()).permute(2, 0, 1)  # Change to C, H, W
            image = image.float() / 255.0 # Normalize image values to 0-1
            label = torch.tensor(label.numpy())
            self.data.append((image, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Convert TensorFlow datasets to PyTorch datasets
train_dataset = ColorectalHistologyDataset(ds_train_tf)
test_dataset = ColorectalHistologyDataset(ds_test_tf)

As usual, we will also need a PyTorch DataLoader for efficient batching and shuffling during training and testing.

Create two variables `train_loader` and `test_loader` by constructing a DataLoader based on the respective dataset. Use a batch size of `128`. Enable shuffling for the `train_loader`, but disable it for the `test_loader`.

In [ ]:
# Create a dataloader around the dataset, specifying the batch size and shuffling
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert isinstance(train_loader, DataLoader)
assert isinstance(test_loader, DataLoader)
assert train_loader.batch_size == 128
assert test_loader.batch_size == 128
assert train_loader.dataset == train_dataset
assert test_loader.dataset == test_dataset


## 3: Take a look at examples

Now that we loaded and split the dataset, let's take a look at an instance.

In our first ML examples, we usually had all data in memory at once (for example, as a Numpy array). Now, we access the data through a data loader, which has functions to load & preprocess the next batch so that we can work with the data (or the CNN can train with it).

A simple way to get one example out of the dataset is to use construct an iterator (`iter`) based on the `train_loader`. With the `next()` function, we get the next batch. This returns a tuple, which you can directly split up into two variables: `example_data` for the image data, and `example_targets` for the respective labels.

In [ ]:
# Predefined code - just execute
# Visualize an example image
image_index = 0
# Get a batch of data and their corresponding labels from the train_loader.
example_data, example_targets = next(iter(train_loader))

Call the `size()` function of `example_data` and print its outputs.

In [ ]:
# Print the size of the example_data
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
assert example_data.size(0) == 128
assert example_data.size(1) == 3
assert example_data.size(2) == 150
assert example_data.size(3) == 150
assert example_targets.size(0) == 128

You should see the batch size of `128` that we have currently loaded from the data loader.
This batch has the dimensions of `3, 150, 150`. This means that we have `150x150px` as the image size, with `3` color channels.

Next, print the element of `example_targets` at the position you previously defined in the variable `image_index`. You should see the target class ID. If you add a conversion to `.item()` at the end, you get it as a normal Python number instead of a tensor.

In [ ]:
# Print the target label of an image
raise NotImplementedError


Next, plot the example image at the index position `image_index`. Refer to the example we did in class for the code. Also, set the title of the plot to the target class.

You can either just plot one color channel of the example image by accessing it directly (e.g., `[image_index][0]`). Alternatively, you can plot the colored image. In this case, you need to reorder the dimensions from `(colorchannels, height, width)` to `(height, with, colorchannels)`. Do this with `.permute(1, 2, 0)` based on the image data before you send it to `imshow()`.

*Hint:* use the `plt` functions `imshow()`, `title()` and `show`().

In [ ]:
# Plot the image with the class as its title
raise NotImplementedError


Finally, let's take a look at the raw data of an image. Simply print `example_data[image_index]` to see the raw data:

In [ ]:
# Print the raw contents of the img variable
raise NotImplementedError


You can see in the output that `img` is a multi-dimensional array. The image contains the intensity values of all three colors (red, green, blue). When creating the PyTorch dataset, we already scaled the pixel values from `0..255` to `0..1`.

## 4: Build the Convolutional Neural Network

Now, we're finally at the step where we define the structure of the neural network! The following would be a good architecture to start with:

![Sample architecture of the CNN](https://github.com/andijakl/MachineLearning/raw/main/lab%203%20-%20deep%20learning%20-%20colorectal%20cancer/lab3b-architecture.png)*Sample architecture of the CNN*

Usually, a CNN has multiple convolutional layers, often followed by a max-pooling layer. At the end, after a flattening layer, two dense (fully connected / linear) layers ensure that the outputs of the convolutional blocks are classified. The last layer has 8 neurons, so that the probability for each of the 8 classes can be predicted.

**Your task:** build an architecture like the one in the image above.

**Some hints:**

* Typical definition of a convolutional layer:  
`nn.Conv2d(3, 32, kernel_size=3)`  
Indiviual parameters:  
  1. Define the **number of inputs** and the **number of neurons**. The first layer has 3 inputs as we are dealing with RGB color images. You could for example start with 32 filters in the first layer, and then go up to 64 layers for the following layers. Refer to the image of the CNN structure above. The number of layers is printed above the image, e.g., `32@148x148` means that we have 3 layers and the images have a shape of `148x148` (missing 1 border pixel on each side due to the `3x3` convolution defined through the `kernel_size`).
  2. **Size of the filters** in this layer, for example `3x3`. Note that the filter size is not visible in the image above. The image size is however indicating the filter size at this stage – it starts at `150x150`, gets a bit smaller with each convolutional layer due to the border pixels. We don't use padding, so a `3x3` filter removes two pixels from the width and height.
* **Activation function:** After every `Conv2d` and Linear layer (except the very last), add a `ReLU` activation function to enable non-linearity.
* Typical definition of a Max-Pool layer:  
`nn.MaxPool2d(2)`  
There is only a single parameter – the size of the pool. Always use a `2x2` pool. This already halves both the width and height of the data, resulting in a 1/4 of the original size. As we start with rather small 150x150px images, you shouldn't go too low too quickly.
* The flattening is simple and doesn't have parameters we need to set:  
`nn.Flatten()`
* The last two layers are dense layers, like:  
`nn.Linear(128, 8)`  
Use 128 nodes for the first dense layer and the `relu` activation function. The second dense layer should have 8 nodes (according to the target classes). The tricky part is calculating the number of input parameters for the first linear layer. You can get it by multiplying the number of filters x the filter width x the filter height.

In [ ]:
# Create a variable "model" and use the Sequential() function to add a list of layers according to the
# description and the image above.
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
# Check if the model is a Sequential model
assert isinstance(model, nn.Sequential), "The model should be a nn.Sequential object"

# Check the number of layers
assert len(list(model.children())) == 13, "The model should have 13 layers"

# Check the first layer (Conv2d)
first_layer = list(model.children())[0]
assert isinstance(first_layer, nn.Conv2d), "First layer should be a Conv2d layer"
assert first_layer.in_channels == 3, "First layer should have 3 input channels"
assert first_layer.out_channels == 32, "First layer should have 32 output channels"

# Check a middle layer (e.g., the second Conv2d)
second_conv_layer = list(model.children())[3]
assert isinstance(second_conv_layer, nn.Conv2d), "Second convolutional layer should be a Conv2d layer"
assert second_conv_layer.in_channels == 32, "Second convolutional layer should have 32 input channels"
assert second_conv_layer.out_channels == 64, "Second convolutional layer should have 64 output channels"

# Check the output layer (Linear)
output_layer = list(model.children())[-1]
assert isinstance(output_layer, nn.Linear), "Output layer should be a Linear layer"
assert output_layer.out_features == 8, "Output layer should have 8 output features"

In [ ]:
# Predefined code - just execute
# Move model to GPU if available
model.to(device)

Next, print the model summary using `torchsummary`. Note the total number of *trainable parameters*.

In [ ]:
# Predefined code - just execute
from torchsummary import summary
summary(model, (3, 150, 150), device=device.type)

How many parameters will need to be trained by our neural network? Create a variable `train_parameters` and store the number as an integer (do not use thousand separators).

In [ ]:
# Create a variable train_parameters and insert the value of trainable params
# from the model summary above
train_parameters = 0
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert train_parameters > 0
assert isinstance(train_parameters, int)

## 5: Loss Function and Optimizer

After the model is defined, you also need to define the loss function and optimizer. Use:

* **Loss function:** `nn.CrossEntropyLoss()`. In PyTorch, the CrossEntropyLoss works directly with logits; there is no need to apply an activation function like Softmax. Variable name: `loss_nf`.
* **Optimizer:** use `optim.Adam()`. Variable name: `optimizer`.

In [ ]:
# Compile the model according to the settings described above
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute this cell.
test_case.assertIsNotNone(loss_fn, "Criterion (loss function) has not been initialized.")
test_case.assertIsNotNone(optimizer, "Optimizer has not been initialized.")

## 6: Training

Prepare a cup of coffee / tea for the next step. Depending on your computer speed, this might take several minutes. After all, we're working with real data and are training a deep neural network.

With the CNN architecture from above, you can still get better results if you train for more epochs - but of course, it'll take a longer time. So let's start low; you can always increase the number of epochs once you know that you're on the right path and everything works.

First, let's define two variables.

* The first is `num_epochs`. Store the number of training epochs you would like to use, e.g., 2 at the beginning. Once you know your code works, re-execute the notebook with a larger epoch count. It's good practice to have this in a variable to make changes in configuration easier in a central place, without directly messing around in code of the for-loop.

* Also define a variable `train_loss_history` as an empty array. We'll append values during training.

In [ ]:
# Configuration: create a variable num_epochs with your config.
# Initialize an empty train_loss_history array.
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert num_epochs > 0
assert len(train_loss_history) == 0

Set the model to **train mode**. We don't evaluate within the loop, so it's enough to only switch to train mode once before the loop. This is important because some layers, like dropout, behave differently during training and evaluation.

In [ ]:
# Set model to training mode
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert model.training

Now it's time for the large training loop. Use the standard procedure for PyTorch:


1. Create a `for`-loop over the range of `num_epochs`.
2. Set a `total_loss` variable to `0`, so that the loss of each epoch can be summed while going through batches.
3. Create an inner `for`-loop over the `train_loader`. It returns two variables that you have as loop variables. Call them: `batch_X`, `batch_y`.
  1. Within the inner loop, first **reset the optimizer** using `zero_grad()`.
  2. Move the `batch_X` and `batch_y` to the `device` where the model is running (CPU / GPU)
  2. Next, send your batch data to the **model** and store its **predictions** in a `y_pred` variable.
  3. Use the predicted and the true labels to calculate the **loss function** you defined above. Store its results in a variable called `loss`.
  4. Call `backward()` on `loss` to compute the gradients of the loss function with respect to all trainable parameters.
  5. **Update the parameters** of the model using `optimizer.step()` using our Adam optimization function you defined before.
  **6. For statistics:** add the `loss.item()` to the total_loss variable so that we can get the sum of all losses of the whole epoch. *Note:* this calculates the average loss per batch, which is sufficient as we want to monitor the trend and not the exact loss per item.

4. Outside of the inner loop: **Compute the average loss**. Divide the `total_loss` by the number of items we used for training (--> `len(train_loader)`).
5. Append the average loss you computed to the `train_loss_history` array.

6. Every epoch, **print** the current epoch number as well as the current training loss.

In [ ]:
# The training loop
raise NotImplementedError


## 7: Visualize the History

Now that your model is trained, let's visualize accuracy and loss.

Execute the following block of code to see the visualization of the training loss. It should decrease during training over the epochs.

In [ ]:
# Plot Training Loss and Test Accuracy
# Pre-defined code
plt.figure(figsize=(10, 4))

plt.plot(train_loss_history, label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.legend()

mid, execution_time = getData()
plt.text(.01, .01, f'{notebook_version}, {mid}, {execution_time}', ha='left', va='bottom', transform=plt.gca().transAxes)

plt.show()

## 8: Evaluate the Model

As the last step, we also want to know how well our model performs. We'll need to evaluate the trained model with the separate test data, which was not used for training. Therefore, the model has never seen it before, making the data a more reliable indicator on the model performance.

First, we need to set up a few things. Create two variables, `correct` and `total`, and set both to 0. We'll use these for counting how many predictions we got correct, compared to the ground truth.

Also, set the `model` to *evaluation mode*. Among other things, this disables dropout features if defined in the structure.

In [ ]:
# Switch the model to evaluation mode
# Initialize the correct and total variables with 0
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert model.training == False
assert correct == 0
assert total == 0

The evaluation is again a loop. This time, we iterate over the `test_loader`. We count the number of correctly classified items.

First, we use the `no_grad` context manager of PyTorch, which disables gradient calculation in its block. This reduces memory consumption for computations. When the code execution leaves the block, it will automatically re-enable gradient calculation.

Within this block, we iterate over the `test_loader` with a `for`-loop, similar to the training loop. As loop variables, we get both the data for the batch (`batch_X`), as well as the labels (`batch_y`).

Within the loop:
1. Move `batch_X` and `batch_y` to the GPU
2. Let the `model` **predict** based on the batch data, and store the results in `y_pred`.
3. **Find the most probable class.** How? With `y_pred.data`, you extract the predicted class labels from the model's output. `torch.argmax` finds the index of the maximum value along dimension 1, which represents the class with the highest probability.
4. **Count how many items are classified correctly.** To do this, compare if `y_pred_class == batch_y`. This would return a vector of 0 or 1, depending on whether the classes are equal (`1`) or not (`0`). To calculate the accuracy, we need to know how many are correct, so use `sum()` to count how many correct predictions we have in that array. Finally, we're now leaving the tensor-world and returning to plain Python numbers, so convert the tensor to a number with `item()`. Add this result to the `correct` variable.
5. The final step is to **count how many items** we have processed in total so far, which will later be required for the percentage. Add `batch_y.size(0)` to the `total` variable.

In [ ]:
# Evaluate accuracy on test set
raise NotImplementedError


In [ ]:
# Assertions for the evaluation part
assert total == 1000, "Total number of test samples should be 1000."

Now we have all the numbers, and we just need to calculate the accuracy: divide correct by total and print the result for our test accuracy. With just two epochs of training, you should reach at least 60% accuracy.

Note that you need to compare that to the baseline accuracy of random guessing. As we have many classification targets, this would be a lot lower than the 60%. If you train the model longer, the accuracy should get a lot better. With 20 epochs, you can reach around 80%. The exact value is different, as it depends on the random split of the training and test set, as well as how neurons were initialized with random weights and what solution the neural network found so far.

In [ ]:
raise NotImplementedError


In [ ]:
assert test_accuracy > 0.60
assert test_accuracy < 0.95

## 9: Test with a new image

As a final step, we will use the trained model to classify completely new images.

Three sample images have been provided, which have been created by Dall-E and already resized to 150x150 pixels to fit our CNN.

Use the following sample images in the code blocks below:

1. https://raw.githubusercontent.com/andijakl/MachineLearning/refs/heads/main/lab%20-%20pytorch%20-%20colorectal%20cancer/dalle-colorectal-tissue-1.png
2. https://raw.githubusercontent.com/andijakl/MachineLearning/refs/heads/main/lab%20-%20pytorch%20-%20colorectal%20cancer/dalle-colorectal-tissue-2.png
3. https://raw.githubusercontent.com/andijakl/MachineLearning/refs/heads/main/lab%20-%20pytorch%20-%20colorectal%20cancer/dalle-colorectal-tissue-3.png

In [ ]:
# Predefined imports - just run this cell
# Additional imports required to download data and work with images in Python
from PIL import Image
import torchvision.transforms as transforms
import requests

Modify the following code block with one of the URLs. You will change this and execute the following blocks several times to get predictions for all three sample images.

In [ ]:
# Modify the following line with the image you want to load in this step
image_url = "https://raw.githubusercontent.com/andijakl/MachineLearning/refs/heads/main/lab%20-%20pytorch%20-%20colorectal%20cancer/dalle-colorectal-tissue-2.png"
image = Image.open(requests.get(image_url, stream=True).raw)

The picture is loaded into the image variable. Use the `imshow()` function of `plt` to show the image. To make sure it is visible, also use `show()`.

In [ ]:
# Show the image you loaded
raise NotImplementedError


For converting the image into a tensor, we need to create a preprocessing stack. This should first resize the image to 150x150 to ensure it is compatible with our model (which can only accept images of this size - the three example images already have the size, but it's always good to make sure the code doesn't crash when someone tries an image with a different size). Additionally, convert the data to a tensor.

To perform this, create a variable called `preprocess`. Use `transforms.Compose()` to supply the stack of transformations to apply with an array `[]`. For the `Resize()` call, supply the image size parameter as a tuple. Afterwards, use `ToTensor()`.

In [ ]:
# Preprocess the image: resize to 150x150 and convert to a tensor
# Store the transform stack in a variable called preprocess
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
test_case.assertIsNotNone(preprocess)
test_case.assertIsInstance(preprocess, transforms.Compose)
test_case.assertEqual(len(preprocess.transforms), 2)
test_case.assertIsInstance(preprocess.transforms[0], transforms.Resize)
test_case.assertEqual(preprocess.transforms[0].size, (150, 150))
test_case.assertIsInstance(preprocess.transforms[1], transforms.ToTensor)

Next, we need to apply the transform to the image to convert it.

There is a little additional detail - the PyTorch stack expects to get a batch of images. If we have a single image, it would have the shape `(Color, Height, Width)`. We need to add a dimension to have a 4D tensor, even if it only contains a single image. After calling `unsqueeze(0)`, the data is transformed to `(1, Color, Height, Width)`.

At the end, the tensor has to be moved to the `device`.

In [ ]:
# Predefined code block, just execute
# Unsqueeeze: adds extra dimension to fit to batches
# Before: tensor has shape    C, H, W
# After:  tensor has shape 1, C, H, W
imageProcessed = preprocess(image).unsqueeze(0).to(device)

In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
test_case.assertEqual(imageProcessed.shape[1], 3, "Image should have 3 color channels (RGB)")
test_case.assertEqual(imageProcessed.shape[2], 150, "Image height should be 150")
test_case.assertEqual(imageProcessed.shape[3], 150, "Image width should be 150")
test_case.assertEqual(imageProcessed.dtype, torch.float32, "Image data type should be float32")

To ensure that our model is in the right mode, activate evaluation mode.

In [ ]:
# Set the model to evaluation mode
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert model.training == False

As a final step, let's ask our model to make a prediction based on the pre-processed image data:

As usual when evaluating, use the `no_grad()` block. Inside, send the image to the model and assign the result to a variable. It's also interesting to print that prediction tensor to see the raw values or how close different predictions are.

Finally, let's find the msot probable class. Use `torch.argmax` from the output to assign what is most likely to a `predicted` variable.

In [ ]:
# Make the prediction
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert predicted is not None, "Predicted class should not be None."
assert isinstance(predicted, torch.Tensor), "Predicted class should be a torch tensor."
assert predicted.shape == (1,), "Predicted class should have a shape of (1,)."
assert predicted.item() in range(8), "Predicted class should be within the valid range of classes (0-7)."

To make it easier, print the predicted class.

Use the `item()` function to convert the tensor variable predicted to a Python number and print it formatted with a short explanation text.

Also, use the `class_names` array from before for a second print statement to convert the numeric class to a class name.

In [ ]:
# Print the prediction
raise NotImplementedError


Modify the following code to insert which class your model predicted each of the three provided sample images:

In [ ]:
# Change the code to contain the predicted class for each of the three samples.
# The numbers 1,2,3 correspond to the png image names provided in the
# text block above.
# To test all three images, scroll back up, change the URL and run the following
# blocks again to get a prediction for the next image.
sample_classes = {
    1: -1,
    2: -1,
    3: -1
}
raise NotImplementedError


In [ ]:
# Pre-defined tests to check your code. Do not modify, just execute.
assert sample_classes[1] > -1
assert sample_classes[2] > -1
assert sample_classes[3] > -1

## 10: Final Checks

To make sure that your whole Jupyter Notebook executes without issues, choose *Runtime -> Restart Session and Run All ...*. Make sure it actually restarts executing all cells – if not, select the command again. Make sure all the lines you wrote execute and all automated tests pass.

Jupyter automatically saves the cell outputs into the notebook itself, so make sure the cell outputs (including the plotted graphs) are visible. To submit your notebook with all the outputs included, follow these steps:
1. Rename the file to include your name, e.g.: "lab_colorectal_cancer_pytorch_25ss_**jakl**.ipynb"
2. Make sure your file is saved (press Ctrl+S)
3. Export the notebook to your computer: File --> Download --> Download .ipynb
4. Go to eCampus and upload your notebook file to the assignment.

## Next Steps (Optional)

When you've reached this point, it's time to hand in your exercise!

But of course, now the fun part could start - trying to figure out ways how to improve the model to achieve better results. You could add additional convolutional & max-pool layers to your model. Or of course, just train for more epochs – even though you might run into overfitting if you train too long, which would then be visible if your training accuracy is far above the test accuracy. You could tweak the number or size of the filters.

But that's outside of the exercise. Make a private copy of the notebook if you wish to modify the network structure or parameters!

*(Note: with this approach, you're optimizing the results for the `ds_test` set, to make its accuracy as high as possible during evaluation. Thus, you're probably overfitting on the test set. To circumvent that, a usual approach is to have training data for learning the model, validation data to improve the model parameters, and use the test set only to get a good real-life estimate of the final model. We didn't go this route in this example, so that we have more examples for the training & testing sets. The main goal of this lab is to get the neural network to work. But if you enjoyed this work so far, there's a whole world of great ways of how you can fine-tune the model, and strategies for preventing overfitting (e.g., dropout) that you can look into. But these are all details – in this lecture, you've come a long way and can be really proud of what you achieved!)*